# 04 - QLoRA模型微调训练

在DGX Spark上使用QLoRA技术对基座模型进行TRIZ领域微调。

**硬件要求**: DGX Spark (128GB Unified Memory) | **训练时间**: ~15小时 (2 epochs)

## 4.1 环境检查与预飞行

验证训练环境是否满足要求：
- 基准评测已完成 (Notebook 03)
- 训练数据已准备
- GPU内存充足 (>60GB)
- 训练配置正确

In [1]:
# 预飞行检查 — TRAIN-01
import sys
sys.path.append('/home/meerkat/mongoose_ai')

import torch
import os
from utils.pipeline_state import PipelineState
from config import (
    BASE_MODEL, MODELS_DIR, DATA_DIR, OUTPUTS_DIR, RESULTS_DIR,
    CHECKPOINTS_DIR, QLORA_CONFIG, DATA_CONFIG
)

state = PipelineState()

print("=" * 60)
print("预飞行检查")
print("=" * 60)

errors = []
warnings_list = []

# 1. 检查基准评测结果
print("\n[1/5] 检查基准评测结果...")
if state.verify("baseline_results"):
    artifact = state.get("baseline_results")
    print(f"  PASS baseline_results: {artifact['path']}")
else:
    warnings_list.append("未找到基准评测结果，建议先运行 Notebook 03")
    print(f"  WARN baseline_results: 未找到 (建议先运行 Notebook 03)")

# 2. 检查训练数据
print("\n[2/5] 检查训练数据...")
if state.verify("processed_dataset"):
    artifact = state.get("processed_dataset")
    print(f"  PASS processed_dataset: {artifact['path']}")
    data_path = artifact['path']
elif state.verify("synthetic_dataset"):
    artifact = state.get("synthetic_dataset")
    print(f"  PASS synthetic_dataset: {artifact['path']}")
    data_path = artifact['path']
else:
    # 回退到默认路径
    data_path = DATA_CONFIG['processed_data_dir']
    if os.path.exists(data_path):
        print(f"  PASS 默认数据路径存在: {data_path}")
    else:
        errors.append(f"训练数据不存在: {data_path}")
        print(f"  FAIL 训练数据: {data_path} 不存在")

# 3. 检查GPU内存
print("\n[3/5] 检查GPU内存...")
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    free_mem = total_mem - torch.cuda.memory_allocated() / (1024**3)
    print(f"  总内存: {total_mem:.1f} GB")
    print(f"  可用内存: {free_mem:.1f} GB")
    if free_mem < 60:
        warnings_list.append(f"可用GPU内存仅 {free_mem:.1f}GB")
        print(f"  WARN 可用内存不足: {free_mem:.1f}GB < 60GB")
    else:
        print(f"  PASS GPU内存充足: {free_mem:.1f}GB")
else:
    errors.append("CUDA不可用")
    print(f"  FAIL CUDA不可用")

# 4. 检查模型路径
print("\n[4/5] 检查模型路径...")
model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])
if os.path.exists(model_path):
    print(f"  PASS 模型路径: {model_path}")
else:
    errors.append(f"模型路径不存在: {model_path}")
    print(f"  FAIL 模型路径: {model_path}")

# 5. 检查训练配置
print("\n[5/5] 检查训练配置...")
lora_dropout = QLORA_CONFIG['lora']['lora_dropout']
target_modules = QLORA_CONFIG['lora']['target_modules']
save_steps = QLORA_CONFIG['training']['save_steps']
save_total_limit = QLORA_CONFIG['training']['save_total_limit']
num_epochs = QLORA_CONFIG['training']['num_train_epochs']
lr = QLORA_CONFIG['training']['learning_rate']
warmup = QLORA_CONFIG['training']['warmup_ratio']

if lora_dropout == 0.0:
    print(f"  PASS lora_dropout=0.0")
else:
    warnings_list.append(f"lora_dropout={lora_dropout} (建议0.0)")
    print(f"  WARN lora_dropout={lora_dropout}")

if isinstance(target_modules, list) and len(target_modules) == 12:
    print(f"  PASS target_modules: {len(target_modules)}个模块")
else:
    warnings_list.append("target_modules数量异常")
    print(f"  WARN target_modules数量异常: {len(target_modules) if isinstance(target_modules, list) else '非列表'}")

print(f"  save_steps={save_steps}, save_total_limit={save_total_limit}")
print(f"  num_train_epochs={num_epochs}, learning_rate={lr}, warmup_ratio={warmup}")
print(f"  lr_scheduler_type={QLORA_CONFIG['training'].get('lr_scheduler_type', 'cosine')}")
print(f"  optim={QLORA_CONFIG['training']['optim']}")

# 总结
print("\n" + "=" * 60)
if errors:
    print(f"预飞行检查失败: {len(errors)}个错误")
    for e in errors:
        print(f"  ERROR: {e}")
    print("=" * 60)
    raise RuntimeError("预飞行检查失败，请修复上述错误后再继续")
else:
    print("预飞行检查通过!")
    if warnings_list:
        print(f"警告: {len(warnings_list)}个")
        for w in warnings_list:
            print(f"  WARN: {w}")
    print("=" * 60)


预飞行检查

[1/5] 检查基准评测结果...
  PASS baseline_results: /home/meerkat/mongoose_ai/results/triz_eval_results_20260604_195346.json

[2/5] 检查训练数据...
  PASS synthetic_dataset: /home/meerkat/mongoose_ai/data/processed

[3/5] 检查GPU内存...
  总内存: 121.7 GB
  可用内存: 121.7 GB
  PASS GPU内存充足: 121.7GB

[4/5] 检查模型路径...
  PASS 模型路径: /home/meerkat/mongoose_ai/models/Qwen3.6-35B-A3B

[5/5] 检查训练配置...
  PASS lora_dropout=0.0
  PASS target_modules: 12个模块
  save_steps=200, save_total_limit=3
  num_train_epochs=2, learning_rate=0.0002, warmup_ratio=0.05
  lr_scheduler_type=cosine
  optim=paged_adamw_8bit

预飞行检查通过!


## 4.2 加载配置和数据

从pipeline_state或默认路径加载处理后的数据集。

In [2]:
# 加载处理后的数据集
from utils.data_utils import load_processed_dataset

print(f"加载数据集: {data_path}")

dataset = load_processed_dataset(data_path)

print(f"\n数据集加载完成!")
for split_name, split_data in dataset.items():
    print(f"  {split_name}: {len(split_data)} 条样本")

# 验证数据集格式
sample = dataset['train'][0] if 'train' in dataset else None
if sample:
    print(f"\n样本字段: {list(sample.keys())}")
    print(f"instruction长度: {len(sample.get('instruction', ''))}")
    print(f"output长度: {len(sample.get('output', ''))}")


INFO:utils.data_utils:加载 train: 2662 条
INFO:utils.data_utils:加载 validation: 313 条
INFO:utils.data_utils:加载 test: 157 条


加载数据集: /home/meerkat/mongoose_ai/data/processed

数据集加载完成!
  train: 2662 条样本
  validation: 313 条样本
  test: 157 条样本

样本字段: ['text', 'subset', 'instruction', 'input', 'output', 'system', 'length']
instruction长度: 36
output长度: 282


## 4.3 加载模型 (4-bit量化)

使用4-bit NF4量化加载模型以节省内存，用于QLoRA训练。

In [3]:
# 加载4-bit量化模型 — TRAIN-07
from utils.training_utils import load_model_and_tokenizer

model_path = os.path.join(MODELS_DIR, BASE_MODEL.split('/')[-1])

print(f"加载模型: {model_path}")
print("启用4-bit量化 (NF4) 以节省内存...")

model, tokenizer = load_model_and_tokenizer(
    model_name_or_path=model_path,
    quantization_config=QLORA_CONFIG['quantization'],
    device_map=‘auto’,
    trust_remote_code=True,
)

print(f"\n模型加载完成!")
print(f"显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


INFO:utils.training_utils:加载模型: /home/meerkat/mongoose_ai/models/Qwen3.6-35B-A3B
INFO:utils.training_utils:设备映射: None


加载模型: /home/meerkat/mongoose_ai/models/Qwen3.6-35B-A3B
启用4-bit量化 (NF4) 以节省内存...


INFO:utils.training_utils:分词器词汇表大小: 248077
INFO:utils.training_utils:启用4-bit量化 (NF4)
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/693 [00:00<?, ?it/s]

/home/meerkat/mongoose_ai/venv_v5/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
INFO:utils.training_utils:模型加载完成
INFO:utils.training_utils:模型参数总量: 33.96B



模型加载完成!
显存占用: 62.61 GB


## 4.4 配置QLoRA

使用显式12模块target_modules列表（非all-linear），lora_dropout=0.0。

In [4]:
# (可选) 验证target_modules配置
from utils.training_utils import find_all_linear_names, get_qwen36_target_modules

print("检测模型中的线性层模块...")
detected = find_all_linear_names(model)
print(f"\n检测到 {len(detected)} 个可用于LoRA的模块")

recommended = get_qwen36_target_modules()
missing = set(recommended) - set(detected)
extra = set(detected) - set(recommended)

if missing:
    print(f"警告: 推荐列表中有但模型中未检测到: {missing}")
if extra:
    print(f"提示: 模型中存在额外模块: {extra}")
if not missing and not extra:
    print("模块列表匹配完美!")


检测模型中的线性层模块...

检测到 13 个可用于LoRA的模块
提示: 模型中存在额外模块: {'shared_expert_gate'}


In [5]:
# 创建QLoRA配置并准备模型
import gc                                                                                                                                                                  
model.config.use_cache = False                                                                                                                                             
gc.collect()                           
torch.cuda.empty_cache()

from peft import get_peft_model                                                                                                                                            
from utils.training_utils import setup_qlora_config                                                                                                                        
                                                                                                                                                                             
lora_config = setup_qlora_config(                                                                                                                                          
    r=QLORA_CONFIG['lora']['r'],                                                                                                                                           
    lora_alpha=QLORA_CONFIG['lora']['lora_alpha'],                                                                                                                         
    target_modules=QLORA_CONFIG['lora']['target_modules'],                                                                                                                 
    lora_dropout=QLORA_CONFIG['lora']['lora_dropout'],                                                                                                                     
    use_rslora=QLORA_CONFIG['lora'].get('use_rslora', False),                                                                                                              
)                                                                                                                                                                          
                                                                                                                                                                             
# Skip prepare_model_for_kbit_training — gradient checkpointing already enabled                                                                                            
model = get_peft_model(model, lora_config)                                                      
model.print_trainable_parameters()                                                                                                                                         
                                                                                                  
print("\nQLoRA配置完成!")   


INFO:utils.training_utils:使用手动指定的target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj_qkv', 'in_proj_z', 'in_proj_b', 'in_proj_a', 'out_proj', 'gate_proj', 'up_proj', 'down_proj']
INFO:utils.training_utils:LoRA配置: rank=64, alpha=128, target_modules=12个
INFO:utils.training_utils:目标模块: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj_qkv', 'in_proj_z', 'in_proj_b', 'in_proj_a', 'out_proj', 'gate_proj', 'up_proj', 'down_proj']


trainable params: 84,664,320 || all params: 34,745,275,008 || trainable%: 0.2437

QLoRA配置完成!


## 4.5 配置训练参数

2 epochs, lr=2e-4, cosine scheduler, 5% warmup, save every 200 steps.

# 热修复 setup_training_arguments — 无需重启kernel                                                                                                                         
import transformers                                                                                                                                                        
from transformers import TrainingArguments                                                                                                                                 
import logging                                                                                                                                                             
                                                                                                                                                                             
logger = logging.getLogger(__name__)                                                                                                                                       
                                         
def setup_training_arguments(                                                                                                                                              
    output_dir: str,                                                                            
    num_train_epochs: int = 2,                                                                                                                                             
    per_device_batch_size: int = 1,                                                                                                                                        
    gradient_accumulation_steps: int = 8,                                                                                                                                  
    learning_rate: float = 2e-4,                                                                                                                                           
    warmup_ratio: float = 0.03,                                                                                                                                            
    save_steps: int = 200,                                                                                                                                                 
    eval_steps: int = 200,                                                                                                                                                 
    logging_steps: int = 10,                                                                                                                                               
    **kwargs                                                                                                                                                               
):                                                                                                                                                                         
    args_dict = {                                                                                                                                                          
          "output_dir": output_dir,                                                               
          "num_train_epochs": num_train_epochs,                                                                                                                              
          "per_device_train_batch_size": per_device_batch_size,                                                                                                              
          "per_device_eval_batch_size": per_device_batch_size,                                                                                                               
          "gradient_accumulation_steps": gradient_accumulation_steps,                                                                                                        
          "learning_rate": learning_rate,                                                                                                                                    
          "warmup_ratio": warmup_ratio,                                                                                                                                      
          "lr_scheduler_type": "cosine",                                                                                                                                     
          "logging_steps": logging_steps,                                                                                                                                    
          "save_steps": save_steps,                                                                                                                                          
          "eval_steps": eval_steps,                                                                                                                                          
          "save_strategy": "steps",                                                                                                                                          
          "eval_strategy": "steps",  # <-- v5 requires explicit eval_strategy                                                                                                
          "logging_strategy": "steps",                                                                                                                                       
          "fp16": True,                                                                                                                                                      
          "bf16": False,                                                                                                                                                     
          "optim": "paged_adamw_8bit",                                                                                                                                       
          "remove_unused_columns": False,                                                                                                                                    
      }                                                                                                                                                                      
    args_dict.update(kwargs)                                                                                                                                               
    training_args = TrainingArguments(**args_dict)                                                                                                                         
    logger.info(f"训练参数配置完成: 输出目录={output_dir}, 轮数={num_train_epochs}, lr={learning_rate}")                                                                   
    return training_args                                                                                                                                                   
                                                                                                                                                                             
# 替换已导入模块中的函数                                                                                                                                                   
import utils.training_utils                                                                                                                                                
utils.training_utils.setup_training_arguments = setup_training_arguments                                                                                                   
                                                                                                                                                                             
print("setup_training_arguments 已热修复")

In [6]:
# 训练参数 — 修改版 (移除与函数硬编码冲突的参数)                                                                                                                           
from utils.training_utils import setup_training_arguments                                                                                                                  
                                                                                                                                                                             
training_args = setup_training_arguments(                                                                                                                                  
    output_dir=QLORA_CONFIG['training']['output_dir'],                                                                                                                     
    num_train_epochs=QLORA_CONFIG['training']['num_train_epochs'],
    per_device_batch_size=QLORA_CONFIG['training']['per_device_train_batch_size'],                                                                                         
    gradient_accumulation_steps=QLORA_CONFIG['training']['gradient_accumulation_steps'],                                                                                   
    learning_rate=QLORA_CONFIG['training']['learning_rate'],                                                                                                               
    warmup_ratio=QLORA_CONFIG['training']['warmup_ratio'],                                                                                                                 
    save_steps=QLORA_CONFIG['training']['save_steps'],                                                                                                                     
    eval_steps=QLORA_CONFIG['training']['eval_steps'],                                                                                                                     
    logging_steps=QLORA_CONFIG['training']['logging_steps'],                                                                                                               
    save_total_limit=QLORA_CONFIG['training']['save_total_limit'],                                                                                                         
    load_best_model_at_end=QLORA_CONFIG['training'].get('load_best_model_at_end', True),                                                                                   
    metric_for_best_model=QLORA_CONFIG['training'].get('metric_for_best_model', 'eval_loss'),                                                                              
    greater_is_better=QLORA_CONFIG['training'].get('greater_is_better', False),                                                                                            
    report_to=QLORA_CONFIG['training'].get('report_to', 'tensorboard'),                                                                                                    
)                                                                                                                                                                      
                                                                                                                                                                             
print("训练参数配置完成!")                                                                                                                                                 
print(f"  输出目录: {training_args.output_dir}")                                                
print(f"  训练轮数: {training_args.num_train_epochs}")                                                                                                                     
print(f"  有效batch_size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")                                                        
print(f"  学习率: {training_args.learning_rate}")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
INFO:utils.training_utils:训练参数配置完成:
INFO:utils.training_utils:  输出目录: /home/meerkat/mongoose_ai/checkpoints/qlora_trtiz_v1
INFO:utils.training_utils:  训练轮数: 2
INFO:utils.training_utils:  有效batch_size: 8
INFO:utils.training_utils:  学习率: 0.0002
INFO:utils.training_utils:  优化器: paged_adamw_8bit


训练参数配置完成!
  输出目录: /home/meerkat/mongoose_ai/checkpoints/qlora_trtiz_v1
  训练轮数: 2
  有效batch_size: 8
  学习率: 0.0002


## 4.6 创建Trainer并开始训练

使用SFTTrainer + formatting_func + packing=True，不传入data_collator。
附加CheckpointValidationCallback验证每个checkpoint。

In [7]:
# 热修复 create_trainer — 移除 max_seq_length                                                                                                                              
import utils.training_utils                                                                                                                                                
from trl import SFTTrainer                                                                                                                                                 
import logging                                                                                                                                                             
                                                                                                                                                                         
logger = logging.getLogger(__name__)                                                                                                                                       
                                                                                              
def create_trainer_v5(model, tokenizer, train_dataset, eval_dataset, training_args,                                                                                        
                    system_message=None, max_seq_length=4096, packing=True):                  
  if system_message is None:                                                                                                                                             
      system_message = (                                                                      
          "You are Meerkat-AI, an expert innovation consultant specializing in TRIZ "                                                                                    
          "(Theory of Inventive Problem Solving). You help users analyze technical contradictions, "                                                                     
          "recommend invention principles, generate innovative solutions, and guide them through "                                                                       
          "the ARIZ algorithm. Always provide structured, actionable advice grounded in TRIZ methodology."                                                               
      )                                                                                                                                                                  
                                                                                                                                                                         
  def formatting_func(example):                                                                                                                                          
      instruction = example.get("instruction", "")                                            
      input_text = example.get("input", "")                                                                                                                              
      output = example.get("output", "")                                                                                                                                 
      sys_msg = example.get("system", system_message)                                                                                                                    
      full_question = f"{instruction}\n\n{input_text}" if input_text else instruction                                                                                    
      messages = [                                                                                                                                                       
          {"role": "system", "content": sys_msg},                                                                                                                        
          {"role": "user", "content": full_question},                                                                                                                    
          {"role": "assistant", "content": output},                                                                                                                      
      ]                                                                                                                                                                  
      return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)                                                                        
                                                                                                                                                                         
  trainer = SFTTrainer(                                                                                                                                                  
      model=model,                                                                                                                                                       
      processing_class=tokenizer,                                                                                                                                        
      train_dataset=train_dataset,                                                                                                                                       
      eval_dataset=eval_dataset,                                                                                                                                         
      args=training_args,                                                                                                                                                
      formatting_func=formatting_func,                                                                                                                                   
      packing=packing,                                                                        
  )                                                                                                                                                                      
                                                                                              
  logger.info("SFTTrainer创建完成")                                                                                                                                      
  return trainer                                                                              
                                                                                                                                                                         
utils.training_utils.create_trainer = create_trainer_v5                                                                                                                    
print("create_trainer 已热修复 (移除 max_seq_length)")

# 检查 SFTTrainer 实际签名                                                                                                                                                 
from trl import SFTTrainer                                                                                                                                                 
import inspect                                                                                                                                                             
                                                                                                                                                                         
sig = inspect.signature(SFTTrainer.__init__)                                                                                                                               
for name, param in sig.parameters.items():                                                                                                                                 
  if name != 'self':                                                                                                                                                     
      default = param.default if param.default is not inspect.Parameter.empty else 'required'                                                                            
      print(f"  {name}: {default}")        

# 热修复 create_trainer — 移除 packing                                                                                                                                     
import utils.training_utils                                                                                                                                                
from trl import SFTTrainer                                                                                                                                               
import logging                                                                                                                                                             
                                                                                              
logger = logging.getLogger(__name__)                                                                                                                                       
                                                                                              
def create_trainer_v5(model, tokenizer, train_dataset, eval_dataset, training_args,                                                                                        
                    system_message=None, max_seq_length=4096, packing=True):                  
  if system_message is None:                                                                                                                                             
      system_message = (                                                                      
          "You are Meerkat-AI, an expert innovation consultant specializing in TRIZ "                                                                                    
          "(Theory of Inventive Problem Solving). You help users analyze technical contradictions, "                                                                     
          "recommend invention principles, generate innovative solutions, and guide them through "                                                                       
          "the ARIZ algorithm. Always provide structured, actionable advice grounded in TRIZ methodology."                                                               
      )                                                                                                                                                                  
                                                                                                                                                                         
  def formatting_func(example):                                                                                                                                          
      instruction = example.get("instruction", "")                                                                                                                     
      input_text = example.get("input", "")                                                                                                                              
      output = example.get("output", "")                                                                                                                                 
      sys_msg = example.get("system", system_message)                                                                                                                    
      full_question = f"{instruction}\n\n{input_text}" if input_text else instruction                                                                                    
      messages = [                                                                                                                                                       
          {"role": "system", "content": sys_msg},                                                                                                                        
          {"role": "user", "content": full_question},                                                                                                                    
          {"role": "assistant", "content": output},                                                                                                                      
      ]                                                                                                                                                                  
      return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)                                                                        
                                                                                                                                                                         
  trainer = SFTTrainer(                                                                                                                                                  
      model=model,                                                                                                                                                       
      processing_class=tokenizer,                                                                                                                                        
      train_dataset=train_dataset,                                                            
      eval_dataset=eval_dataset,                                                                                                                                         
      args=training_args,
      formatting_func=formatting_func,                                                                                                                                   
  )                                                                                           
                                                                                                                                                                         
  logger.info("SFTTrainer创建完成")                                                           
  return trainer                                                                                                                                                         
                                                                                                                                                                         
utils.training_utils.create_trainer = create_trainer_v5                                                                                                                    
print("create_trainer 已热修复 (移除 packing)")    

# 热修复 training_args — 禁用 fp16 避免 GradScaler + BF16 冲突                                                                                                             
training_args.fp16 = False                                                                                                                                                 
training_args.bf16 = False  # 也让它保持 False                                                                                                                             
print(f"fp16={training_args.fp16}, bf16={training_args.bf16}")

create_trainer 已热修复 (移除 max_seq_length)
  model: required
  args: None
  data_collator: None
  train_dataset: None
  eval_dataset: None
  processing_class: None
  compute_loss_func: None
  compute_metrics: None
  callbacks: None
  optimizers: (None, None)
  optimizer_cls_and_kwargs: None
  preprocess_logits_for_metrics: None
  peft_config: None
  formatting_func: None
create_trainer 已热修复 (移除 packing)
fp16=False, bf16=False


In [8]:
# 创建Trainer并训练 — TRAIN-01, TRAIN-04, TRAIN-09
from utils.training_utils import create_trainer, CheckpointValidationCallback

# 创建checkpoint验证回调
checkpoint_callback = CheckpointValidationCallback(
    tokenizer=tokenizer,
    test_prompt="请解释TRIZ的分割原理及其应用场景。",
)

trainer = create_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    training_args=training_args,
    system_message=DATA_CONFIG['chatml']['system_message'],
    max_seq_length=DATA_CONFIG['chatml']['max_length'],
    packing=True,
)

# 添加验证回调
trainer.add_callback(checkpoint_callback)

print("Trainer创建完成 (含checkpoint验证回调)")
print("=" * 60)

# 开始训练
trainer.train()

print("=" * 60)
print("训练完成!")

# 打印验证结果摘要
if checkpoint_callback.validation_results:
    print(f"\nCheckpoint验证结果 ({len(checkpoint_callback.validation_results)}个):")
    passed = sum(1 for r in checkpoint_callback.validation_results if r.get('status') == 'PASSED')
    failed = sum(1 for r in checkpoint_callback.validation_results if r.get('status') == 'FAILED')
    print(f"  通过: {passed}, 失败: {failed}")
    for r in checkpoint_callback.validation_results:
        status = r.get('status', 'UNKNOWN')
        print(f"    step={r['step']}: {status}")


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Applying formatting function to train dataset:   0%|          | 0/2662 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/2662 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2662 [00:00<?, ? examples/s]

Applying formatting function to eval dataset:   0%|          | 0/313 [00:00<?, ? examples/s]

...

INFO:__main__:SFTTrainer创建完成
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'bos_token_id': None, 'pad_token_id': 248044}.


Trainer创建完成 (含checkpoint验证回调)


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/telemetry/trl/SFTTrainer "HTTP/1.1 200 OK"


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,1.509895,1.417523,1.441125,323153.000000,0.707562
400,1.238283,1.404877,1.253900,646980.000000,0.709939
600,1.091870,1.398367,1.231966,970003.000000,0.711564
666,1.178837,1.397888,1.231305,1077178.000000,0.711441


[CHECKPOINT] PASSED: step=200, size=161.6MB, loss=4.8688
[CHECKPOINT] PASSED: step=400, size=161.6MB, loss=4.9065
[CHECKPOINT] PASSED: step=600, size=161.6MB, loss=4.9281
[CHECKPOINT] PASSED: step=666, size=161.6MB, loss=4.9577
训练完成!

Checkpoint验证结果 (4个):
  通过: 4, 失败: 0
    step=200: PASSED
    step=400: PASSED
    step=600: PASSED
    step=666: PASSED


In [9]:
# 1. 禁用自动加载最佳模型                                                                                                                                                  
training_args.load_best_model_at_end = False                                                                                                                               
                                                                                                                                                                         
# 2. 手动保存最终适配器                                                                                                                                                    
from utils.training_utils import save_adapter_only                                                                                                                         
import os                                                                                                                                                                  
                                                                                              
adapter_output_dir = os.path.join(MODELS_DIR, 'meerkat_triz_adapter_v1')                                                                                                   
                                                                                              
# 保存当前模型状态                                                                                                                                                         
save_adapter_only(model, tokenizer, adapter_output_dir)                                         
                                                                                                                                                                         
print(f"\n适配器已保存到: {adapter_output_dir}")                                                                                                                           
for f in os.listdir(adapter_output_dir):                                                                                                                                   
  fp = os.path.join(adapter_output_dir, f)                                                                                                                               
  if os.path.isfile(fp):                                                                                                                                                 
      size = os.path.getsize(fp) / (1024**2)                                                                                                                             
      print(f"  {f}: {size:.2f} MB")                                                                                                                                     
                                                                                                                                                                         
# If you want to identify which checkpoint was best, check the checkpoint folders:                                                                                           
                                                                                                                                                                         
import json                                                                                                                                                                
                                                                                                                                                                         
checkpoint_dir = QLORA_CONFIG['training']['output_dir']
checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith('checkpoint-')]                                                                                       
                                                                                                                                                                         
best_loss = float('inf')               
best_ckpt = None                                                                                                                                                           
for ckpt in sorted(checkpoints):                                                                                                                                           
  trainer_state_path = os.path.join(checkpoint_dir, ckpt, 'trainer_state.json')
  if os.path.exists(trainer_state_path):                                                                                                                                 
      with open(trainer_state_path) as f:                                                                                                                                
          state = json.load(f)                                                                                                                                           
      # Find best eval loss in this checkpoint's history                                                                                                                 
      for log in state.get('log_history', []):                                                                                                                           
          if 'eval_loss' in log and log['eval_loss'] < best_loss:                                                                                                        
              best_loss = log['eval_loss']                                                                                                                               
              best_ckpt = ckpt                                                                                                                                           
                                                                                                                                                                         
print(f"最佳checkpoint: {best_ckpt} (eval_loss={best_loss:.4f})")

INFO:utils.training_utils:保存LoRA适配器到: /home/meerkat/mongoose_ai/models/meerkat_triz_adapter_v1
INFO:utils.training_utils:适配器保存完成! 元数据: {'adapter_type': 'LORA', 'base_model': '/home/meerkat/mongoose_ai/models/Qwen3.6-35B-A3B', 'timestamp': '2026-06-19T15:29:55.968963', 'sha256': '1f909cb016c550b3ddfd5c123b81e1c45b18d4b81cecd777899c31c7712fef54'}



适配器已保存到: /home/meerkat/mongoose_ai/models/meerkat_triz_adapter_v1
  chat_template.jinja: 0.01 MB
  tokenizer.json: 19.06 MB
  tokenizer_config.json: 0.00 MB
  adapter_info.json: 0.00 MB
  adapter_config.json: 0.00 MB
  README.md: 0.01 MB
  adapter_model.safetensors: 161.57 MB
最佳checkpoint: checkpoint-666 (eval_loss=1.3979)


## 4.7 保存适配器

保存LoRA适配器（仅100-200MB）并注册到pipeline_state。
元数据包括训练步数、最终loss、SHA-256哈希。

In [10]:
# 保存LoRA适配器 — TRAIN-08
from utils.training_utils import save_adapter_only
import json
from utils.pipeline_state import PipelineState                                           
state = PipelineState()
adapter_output_dir = os.path.join(MODELS_DIR, 'meerkat_triz_adapter_v1')

# 收集训练元数据
training_metadata = {
    'training_steps': trainer.state.global_step,
    'num_train_epochs': QLORA_CONFIG['training']['num_train_epochs'],
    'learning_rate': QLORA_CONFIG['training']['learning_rate'],
    'final_loss': trainer.state.log_history[-1].get('loss', 'N/A') if trainer.state.log_history else 'N/A',
    'best_eval_loss': trainer.state.best_metric if hasattr(trainer.state, 'best_metric') else 'N/A',
    'checkpoint_validation': checkpoint_callback.validation_results if 'checkpoint_callback' in dir() else [],
}

save_adapter_only(model, tokenizer, adapter_output_dir, metadata=training_metadata)

print(f"\n适配器已保存到: {adapter_output_dir}")

# 查看文件
for f in os.listdir(adapter_output_dir):
    fp = os.path.join(adapter_output_dir, f)
    if os.path.isfile(fp):
        size = os.path.getsize(fp) / (1024**2)
        print(f"  {f}: {size:.2f} MB")

# 读取并显示元数据
info_path = os.path.join(adapter_output_dir, 'adapter_info.json')
if os.path.exists(info_path):
    with open(info_path, 'r') as f:
        info = json.load(f)
    print(f"\n适配器元数据:")
    for k, v in info.items():
        print(f"  {k}: {v}")

# 注册到pipeline_state
state.register(
    name='adapter_checkpoint',
    path=adapter_output_dir,
    artifact_type='model',
    metadata={
        'base_model': BASE_MODEL,
        'training_steps': trainer.state.global_step,
        'final_loss': trainer.state.log_history[-1].get('loss', 'N/A') if trainer.state.log_history else 'N/A',
        'adapter_dir': adapter_output_dir,
    }
)

print(f"\n适配器已注册到 pipeline_state")


INFO:utils.training_utils:保存LoRA适配器到: /home/meerkat/mongoose_ai/models/meerkat_triz_adapter_v1
INFO:utils.training_utils:适配器保存完成! 元数据: {'adapter_type': 'LORA', 'base_model': '/home/meerkat/mongoose_ai/models/Qwen3.6-35B-A3B', 'timestamp': '2026-06-19T15:29:56.387752', 'training_steps': 666, 'num_train_epochs': 2, 'learning_rate': 0.0002, 'final_loss': 'N/A', 'best_eval_loss': 1.3978878259658813, 'checkpoint_validation': [{'step': 200, 'timestamp': '2026-06-19T14:10:47.266643', 'size_mb': 161.57, 'status': 'PASSED', 'loss': 4.8688}, {'step': 400, 'timestamp': '2026-06-19T14:44:09.786352', 'size_mb': 161.57, 'status': 'PASSED', 'loss': 4.9065}, {'step': 600, 'timestamp': '2026-06-19T15:17:32.820125', 'size_mb': 161.57, 'status': 'PASSED', 'loss': 4.9281}, {'step': 666, 'timestamp': '2026-06-19T15:29:55.345464', 'size_mb': 161.57, 'status': 'PASSED', 'loss': 4.9577}], 'sha256': '1f909cb016c550b3ddfd5c123b81e1c45b18d4b81cecd777899c31c7712fef54'}
INFO:utils.pipeline_state:注册工件: adapter_chec


适配器已保存到: /home/meerkat/mongoose_ai/models/meerkat_triz_adapter_v1
  chat_template.jinja: 0.01 MB
  tokenizer.json: 19.06 MB
  tokenizer_config.json: 0.00 MB
  adapter_info.json: 0.00 MB
  adapter_config.json: 0.00 MB
  README.md: 0.01 MB
  adapter_model.safetensors: 161.57 MB

适配器元数据:
  adapter_type: LORA
  base_model: /home/meerkat/mongoose_ai/models/Qwen3.6-35B-A3B
  timestamp: 2026-06-19T15:29:56.387752
  training_steps: 666
  num_train_epochs: 2
  learning_rate: 0.0002
  final_loss: N/A
  best_eval_loss: 1.3978878259658813
  checkpoint_validation: [{'step': 200, 'timestamp': '2026-06-19T14:10:47.266643', 'size_mb': 161.57, 'status': 'PASSED', 'loss': 4.8688}, {'step': 400, 'timestamp': '2026-06-19T14:44:09.786352', 'size_mb': 161.57, 'status': 'PASSED', 'loss': 4.9065}, {'step': 600, 'timestamp': '2026-06-19T15:17:32.820125', 'size_mb': 161.57, 'status': 'PASSED', 'loss': 4.9281}, {'step': 666, 'timestamp': '2026-06-19T15:29:55.345464', 'size_mb': 161.57, 'status': 'PASSED', 'loss

## 4.8 Checkpoint恢复 (可选)

如果训练中断，运行此单元格从最新checkpoint恢复。
验证LR scheduler连续性。

In [11]:
# Checkpoint恢复 — TRAIN-10
# 如果训练中断，取消注释并运行此单元格

# from utils.training_utils import resume_from_checkpoint
# 
# checkpoint_dir = QLORA_CONFIG['training']['output_dir']
# 
# # 查找最新checkpoint
# latest_checkpoint = None
# if os.path.exists(checkpoint_dir):
#     checkpoints = [d for d in os.listdir(checkpoint_dir) 
#                    if d.startswith('checkpoint-') and os.path.isdir(os.path.join(checkpoint_dir, d))]
#     if checkpoints:
#         latest = sorted(checkpoints, key=lambda x: int(x.split('-')[1]))[-1]
#         latest_checkpoint = os.path.join(checkpoint_dir, latest)
# 
# if latest_checkpoint:
#     print(f'找到最新checkpoint: {latest_checkpoint}')
#     print('恢复训练中...')
#     
#     resume_info = resume_from_checkpoint(trainer, latest_checkpoint)
#     
#     print(f'\n恢复验证:')
#     print(f'  恢复前step: {resume_info["initial_step"]}')
#     print(f'  恢复后step: {resume_info["resumed_step"]}')
#     print(f'  恢复前lr: {resume_info["initial_lr"]:.2e}')
#     print(f'  恢复后lr: {resume_info["resumed_lr"]:.2e}')
#     
#     if resume_info['resumed_step'] > resume_info['initial_step']:
#         print('  ✓ 恢复成功: step已增加')
#     else:
#         print('  ✗ 恢复异常: step未增加')
# else:
#     print('未找到checkpoint')


## 4.9 清理显存

释放GPU内存。

In [12]:
# 清理显存
del model
del tokenizer
del trainer
if 'checkpoint_callback' in dir(): del checkpoint_callback
torch.cuda.empty_cache()

print("显存已清理")
print(f"当前显存占用: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


显存已清理
当前显存占用: 1.69 GB


---

## 下一步

微调完成！请打开: **05_model_evaluation.ipynb** 评估微调效果